# Energy Predeiction From Power Plant Dataset=

# 1. Importing Required Libaries 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

# 2. Importing dataset and bit Visualisation 

In [ ]:
df = pd.read_csv("dataset/energy_powerplant_data.csv")

In [ ]:
df.head() 

In [ ]:
df.info()

# 3. Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("PE", axis=1)
Y = df["PE"]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 4. Scalling 

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

# 5. Using Pytorch for ANN

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype=torch.float32).view(-1, 1)

# 6. DataSet and DataLoader

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 7. ANN Model (Deep Learning)

In [ ]:

# Define our ANN model
class EnergyPredictor(nn.Module):
    def __init__(self, input_size):
        super(EnergyPredictor, self).__init__()

        self.model = nn.Sequential(

            # first hidden layer
            nn.Linear(input_size, 6),
            nn.ReLU(),

            # second hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # output layer
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)
    

In [ ]:
import torch.optim as optim

model = EnergyPredictor(X_train.shape[1]) # input size is number of features

criterion = nn.MSELoss() # loss function for regression
optimizer = optim.Adam(model.parameters()) # optimizer to update model parameters

# 8. Traning ANN model

In [ ]:
training_losses = []  
val_losses = []

best_val_loss = float('inf') # initialize best validation loss to infinity

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for Xb, Yb in train_loader:

        optimizer.zero_grad() # clear previous gradients

        predictions = model(Xb) # predict output for current batch
        loss = criterion(predictions, Yb) # compute loss
        loss.backward() # backpropagation to compute gradients
        optimizer.step() # update model parameters

        running_loss += loss.item() # loss is a tensor, we need to get its value using .item()

    epoch_loss = running_loss / len(train_loader) # average loss for the epoch
    training_losses.append(epoch_loss) # store the loss for plotting later

    # validation step (optional, can be done every few epochs)

    running_val_loss = 0.0
    model.eval() # set model to evaluation mode

    with torch.no_grad(): # disable gradient computation for validation
        for Xb, Yb in test_loader:
            val_predictions = model(Xb)
            val_loss = criterion(val_predictions, Yb)
            running_val_loss += val_loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss) # store validation loss for plotting
  
    # Check if this is the best validation loss so far
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "model/best_model.pt") # save the best model parameters

In [ ]:
# load the best model parameters 
model.load_state_dict(torch.load("model/best_model.pt"))

# 9. Model Evaluation 

In [ ]:
# evaluate on test set
model.eval()
with torch.no_grad():
    train_predictions = model(X_train_tensor)
    train_loss = criterion(train_predictions, Y_train_tensor)
    print(f"Train MSE Loss: {train_loss.item()}")
    
    test_predictions = model(X_test_tensor)
    test_loss = criterion(test_predictions, Y_test_tensor)
    print(f"Test MSE Loss: {test_loss.item()}")

    

In [ ]:
from sklearn.metrics import r2_score

print(f"Train R^2 Score: {r2_score(Y_train_tensor.numpy(), train_predictions.numpy())}")
print(f"Test R^2 Score: {r2_score(Y_test_tensor.numpy(), test_predictions.numpy())}")

In [ ]:
predicted_df = pd.DataFrame(test_predictions.numpy(), columns=["Predicted_PE"])
actual_df = pd.DataFrame(Y_test_tensor.numpy(), columns=["Actual_PE"])

comparison_df = pd.concat([actual_df, predicted_df], axis=1)
comparison_df.head()

# 10. Visuslation of the ANN loss 

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(training_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss over Epochs')
plt.legend()  

# 11. Saving the model 

In [ ]:
import pickle

with open("model/ANN_energy_predictor.pkl", "wb") as f:
    pickle.dump(model.state_dict(), f)

# 12. Requirement File for Package Version 

In [ ]:
import pkg_resources

# List of the packages you know you're using
required_packages = [
    'numpy',
    'pandas',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'ipykernel',
    'pickel',
    'torch',
]

requirements = []

for package in required_packages:
    try:
        version = pkg_resources.get_distribution(package).version
        requirements.append(f"{package}=={version}")
    except pkg_resources.DistributionNotFound:
        print(f"Package {package} not found in the environment.")

#requirements to a file
with open('requirements.txt', 'w') as f:
    for line in requirements:
        f.write(line + '\n')